# Final Model Evaluation

Comparison of four phishing URL detection models:
- Logistic Regression
- Support Vector Machine (linear kernel)
- Random Forest
- XGBoost

**Training dataset:** url_dataset.csv (approximately 450k URLs)  
**Inference dataset:** PhiUSIIL_Phishing_URL_Dataset.csv (approximately 235k URLs)  
**Label convention:** 0 = phishing, 1 = legitimate

In [ ]:
import time
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src import data_loader, feature_extractor, metrics
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

## 1. Data Loading

In [ ]:
# Training data (url_dataset)
X_train, X_val, X_test, y_train, y_val, y_test = data_loader.load_data()

# Inference data (PhiUSIIL)
inference_df = pd.read_csv('../data/PhiUSIIL_Phishing_URL_Dataset.csv')
X_inference, y_inference = feature_extractor.extract_features_from_dataset(
    inference_df, url_col='URL', label_col='label'
)
y_inference = y_inference['label']

# Scaled versions for Logistic Regression and Support Vector Machine
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
X_inference_scaled = scaler.transform(X_inference)

print(f"Train: {X_train.shape}")
print(f"Validation:   {X_val.shape}")
print(f"Test:  {X_test.shape}")
print(f"Inference: {X_inference.shape}")

## 2. Load Saved Models

Models are saved locally under `saved_models/`. Notebooks 01 to 04 should be run before this step.

In [ ]:
# Load saved models
with open('../saved_models/logistic_regression_model_v1.pkl', 'rb') as f:
    lr = pickle.load(f)

with open('../saved_models/support_vector_machine_model_v1.pkl', 'rb') as f:
    svm = pickle.load(f)

rf = joblib.load('../saved_models/random_forest_v1.joblib')

xgb = XGBClassifier()
xgb.load_model('../saved_models/xgboost_model_v1.json')

models = {
    'Logistic Regression': ('scaled', lr),
    'Support Vector Machine': ('scaled', svm),
    'Random Forest': ('raw', rf),
    'XGBoost': ('raw', xgb),
}

print("All models loaded.")

## 3. Performance: Training Dataset (Test Split)

In [ ]:
test_results = []

for name, (dtype, model) in models.items():
    X = X_test_scaled if dtype == 'scaled' else X_test
    y_pred = model.predict(X)
    test_results.append({
        'Model': name,
        'Accuracy': metrics.accuracy(y_test, y_pred),
        'Precision': metrics.precision(y_test, y_pred),
        'Recall': metrics.recall(y_test, y_pred),
        'F1 Score': metrics.f1_score(y_test, y_pred),
    })

test_df = pd.DataFrame(test_results).set_index('Model')
print(test_df.round(4).to_string())

## 4. Performance: Inference Dataset (PhiUSIIL)

In [ ]:
inf_results = []

for name, (dtype, model) in models.items():
    X = X_inference_scaled if dtype == 'scaled' else X_inference
    y_pred = model.predict(X)
    inf_results.append({
        'Model': name,
        'Accuracy': metrics.accuracy(y_inference, y_pred),
        'Precision': metrics.precision(y_inference, y_pred),
        'F1 Score': metrics.f1_score(y_inference, y_pred),
    })

inf_df = pd.DataFrame(inf_results).set_index('Model')
print(inf_df.round(4).to_string())

## 5. Which Model Trains and Predicts the Fastest?

Training times are recorded from notebook runs (01-04) and saved to `saved_models/training_times.json`.

In [ ]:
# Load training times from individual notebook runs
import json
with open('../saved_models/training_times.json', 'r') as f:
    train_times = json.load(f)

# Prediction speed: time to classify 1000 URLs
pred_times = {}
for name, (dtype, model) in models.items():
    X_batch = (X_test_scaled if dtype == 'scaled' else X_test)[:1000]
    start = time.time()
    for _ in range(10):
        model.predict(X_batch)
    pred_times[name] = (time.time() - start) / 10  # average over 10 runs

time_df = pd.DataFrame({
    'Training Time (s)': train_times,
    'Prediction Time (1k URLs, s)': pred_times,
})
print(time_df.round(4).to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(list(train_times.keys()), list(train_times.values()))
axes[0].set_xlabel('Seconds')
axes[0].set_title('Training Time')

axes[1].barh(list(pred_times.keys()), list(pred_times.values()))
axes[1].set_xlabel('Seconds')
axes[1].set_title('Prediction Time (1k URLs)')

plt.tight_layout()
plt.show()

## 6. Which Model Is the Lightest?

In [ ]:
saved_models = {
    'Logistic Regression': '../saved_models/logistic_regression_model_v1.pkl',
    'SVM': '../saved_models/support_vector_machine_model_v1.pkl',
    'Random Forest': '../saved_models/random_forest_v1.joblib',
    'XGBoost': '../saved_models/xgboost_model_v1.json',
}

model_sizes = {name: os.path.getsize(path) for name, path in saved_models.items()}

size_df = pd.DataFrame({
    'Size (KB)': {k: v / 1024 for k, v in model_sizes.items()},
})
print(size_df.round(2).to_string())

## 7. Which Features Matter the Most to Each Model?

In [ ]:
feature_names = X_train.columns.tolist()

# Extract importances (normalized to sum to 1)
lr_imp = np.abs(lr.coef_[0])
lr_imp = lr_imp / lr_imp.sum()

svm_imp = np.abs(svm.coef_[0])
svm_imp = svm_imp / svm_imp.sum()

rf_imp = rf.feature_importances_
rf_imp = rf_imp / rf_imp.sum()

xgb_imp = xgb.feature_importances_
xgb_imp = xgb_imp / xgb_imp.sum()

imp_df = pd.DataFrame({
    'Logistic Regression': lr_imp,
    'Support Vector Machine': svm_imp,
    'Random Forest': rf_imp,
    'XGBoost': xgb_imp,
}, index=feature_names)

print(imp_df.round(4).to_string())

# Top 5 features per model
print("\nTop 5 Features per Model:")
for col in imp_df.columns:
    top5 = imp_df[col].sort_values(ascending=False).head(5)
    print(f"\n  {col}:")
    for feat, val in top5.items():
        print(f"    {feat}: {val:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flat, imp_df.columns):
    sorted_imp = imp_df[col].sort_values(ascending=True)
    ax.barh(sorted_imp.index, sorted_imp.values)
    ax.set_title(col)
    ax.set_xlabel('Normalized Importance')

plt.suptitle('Feature Importance by Model', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. How Well Do the Models Protect Users?

In phishing detection, **not all errors are equal**:
- A phishing URL classified as legitimate (**missed phishing**) is a security breach since a user could get phished.
- A legitimate URL classified as phishing (**false alarm**) is an inconvenience, but not a security risk.

So, **phishing recall** (the proportion of phishing URLs correctly detected) is the most critical security metric in our task.

In [ ]:
print("=" * 60)
print("SECURITY METRICS — Training Dataset (Test Split)")
print("=" * 60)

sec_test = []
for name, (dtype, model) in models.items():
    X = X_test_scaled if dtype == 'scaled' else X_test
    y_pred = model.predict(X)
    phishing_recall = metrics.recall(y_test, y_pred, pos_label=0)
    missed_phishing = 1 - phishing_recall
    legitimate_recall = metrics.recall(y_test, y_pred, pos_label=1)
    false_alarm_rate = 1 - legitimate_recall
    sec_test.append({
        'Model': name,
        'Phishing Recall': phishing_recall,
        'Missed Phishing Rate': missed_phishing,
        'Legitimate Recall': legitimate_recall,
        'False Alarm Rate': false_alarm_rate,
    })

sec_test_df = pd.DataFrame(sec_test).set_index('Model')
print(sec_test_df.round(4).to_string())

print("\n" + "=" * 60)
print("SECURITY METRICS — Inference Dataset (PhiUSIIL)")
print("=" * 60)

sec_inf = []
for name, (dtype, model) in models.items():
    X = X_inference_scaled if dtype == 'scaled' else X_inference
    y_pred = model.predict(X)
    phishing_recall = metrics.recall(y_inference, y_pred, pos_label=0)
    missed_phishing = 1 - phishing_recall
    legitimate_recall = metrics.recall(y_inference, y_pred, pos_label=1)
    false_alarm_rate = 1 - legitimate_recall
    sec_inf.append({
        'Model': name,
        'Phishing Recall': phishing_recall,
        'Missed Phishing Rate': missed_phishing,
        'Legitimate Recall': legitimate_recall,
        'False Alarm Rate': false_alarm_rate,
    })

sec_inf_df = pd.DataFrame(sec_inf).set_index('Model')
print(sec_inf_df.round(4).to_string())

In [ ]:
# Confusion matrices for inference dataset
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, (dtype, model)) in zip(axes, models.items()):
    X = X_inference_scaled if dtype == 'scaled' else X_inference
    y_pred = model.predict(X)
    confusion_matrix = metrics.compute_confusion_matrix(y_inference, y_pred)
    tp, fp = confusion_matrix[0]
    fn, tn = confusion_matrix[1]
    # Reorder to [phishing, legitimate] for display
    confusion_matrix_display = np.array([[tn, fp], [fn, tp]])
    sns.heatmap(confusion_matrix_display, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Phishing', 'Legitimate'],
                yticklabels=['Phishing', 'Legitimate'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Inference Dataset (PhiUSIIL)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Can Attackers Bypass the Models?

From an attacker's perspective, some URL features are easier to manipulate than others when crafting a phishing URL. A model that relies heavily on easily-manipulated features is more vulnerable to evasion.

| Controllability | Features | Reasoning |
|----------------|----------|----------|
| **Easy** | `IsHTTPS`, `URLLength`, `NoOfLettersInURL`, `NoOfDegitsInURL` | Attacker fully controls the URL they craft. Free HTTPS certs (Let's Encrypt) make `IsHTTPS` trivial to fake. |
| **Medium** | `DomainLength`, `TLDLength`, `NoOfSubDomain`, `NoOfEqualsInURL`, `NoOfQMarkInURL`, `NoOfAmpersandInURL`, `NoOfObfuscatedChar`, `HasObfuscation`, `NoOfOtherSpecialCharsInURL`, `IsDomainIP` | Attacker controls these but changing them may affect phishing effectiveness or they are constrained by URL structure. |
| **Hard** | `CharContinuationRate`, `ObfuscationRatio`, `LetterRatioInURL`, `DegitRatioInURL`, `SpacialCharRatioInURL` | These are ratios and emergent properties — manipulating one shifts the others, making them hard to control independently. |

In [ ]:
easy = ['IsHTTPS', 'URLLength', 'NoOfLettersInURL', 'NoOfDegitsInURL']
medium = ['DomainLength', 'TLDLength', 'NoOfSubDomain',
          'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
          'NoOfObfuscatedChar', 'HasObfuscation', 'NoOfOtherSpecialCharsInURL',
          'IsDomainIP']
hard = ['CharContinuationRate', 'ObfuscationRatio',
        'LetterRatioInURL', 'DegitRatioInURL', 'SpacialCharRatioInURL']

robustness = {}
for col in imp_df.columns:
    easy_weight = imp_df.loc[easy, col].sum()
    medium_weight = imp_df.loc[medium, col].sum()
    hard_weight = imp_df.loc[hard, col].sum()
    robustness[col] = {
        'Easy to manipulate': easy_weight,
        'Medium': medium_weight,
        'Hard to manipulate': hard_weight,
    }

rob_df = pd.DataFrame(robustness)
print(rob_df.round(4).to_string())

# Visualization
rob_df.T.plot(kind='bar', stacked=True, figsize=(10, 5),
              color=['#e74c3c', '#f39c12', '#27ae60'])
plt.ylabel('Proportion of Total Importance')
plt.title('Model Reliance on Feature Controllability')
plt.legend(title='Attacker Controllability')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("A higher green bar = model relies more on hard-to-manipulate features = more robust.")
print("A higher red bar = model relies on easily-faked features = more vulnerable to evasion.")